In [1]:
# Library imports
import torch
import math
import random
import json
import nltk
import itertools
import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

c:\Users\erwin\miniconda3\envs\contradetect\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Determine what device to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    device_name = torch.cuda.get_device_name(device)

    print(f'CUDA on {device_name}')
else:
    print(f'CPU')

CUDA on NVIDIA RTX 500 Ada Generation Laptop GPU


In [3]:
# Check if everything's cleaned
allocated = torch.cuda.memory_allocated() / (1024 * 1024)
reserved = torch.cuda.memory_reserved() / (1024 * 1024)

print(f'Allocated: {allocated:.1f} MB; Reserved: {reserved:.1f} MB')

Allocated: 0.0 MB; Reserved: 0.0 MB


In [4]:
NLI_MODEL = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli' # https://huggingface.co/MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2' # https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2
NLTK_MODEL = 'tokenizers/punkt/english.pickle'

In [5]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\erwin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [6]:
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)
embed_model = SentenceTransformer(EMBED_MODEL).to(device)

Loading weights: 100%|██████████| 394/394 [00:00<00:00, 4926.77it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 41230.51it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
dataset = json.load(open('datasets/ContraDoc/ContraDoc.json', 'r'))

print(f'Dataset loaded with {len(dataset["pos"])} positive and {len(dataset["neg"])} negative examples.')

Dataset loaded with 449 positive and 442 negative examples.


In [ ]:

def get_closest_sentence_pairs(sentences, embeddings, top_k=None):
    """
    Given a list of sentences and their embeddings, return sentence pairs with closest embeddings.
    
    Args:
        sentences: list of sentence strings
        embeddings: numpy array of embeddings for each sentence
        top_k: number of closest pairs to return
    
    Returns:
        list of tuples (sentence_a, sentence_b, similarity_score)
    """
    
    # Compute pairwise cosine similarity
    similarity_matrix = cosine_similarity(embeddings)
    
    for i in range(len(sentences)):
        similarity_matrix[i][i] = -1.0  # Exclude self-similarity by setting diagonal to -1

    pairs = [ (i, np.argmax(similarity_matrix[i])) for i in range(len(sentences)) ]
    
    # Sort by similarity score (descending) and get top-k
    if top_k is not None:
        pairs.sort(key=lambda x: similarity_matrix[x[0]][x[1]], reverse=True)
        pairs = pairs[:top_k]
        
    # Return pairs with sentences and similarity scores
    return [(sentences[i], sentences[j], similarity_matrix[i][j]) for i, j in pairs]

def predict(premise, hypothesis):
    input = nli_tokenizer(premise, hypothesis, truncation=True, return_tensors="pt").to(device)

    output = nli_model(input["input_ids"])
    prediction = torch.softmax(output["logits"][0], -1).tolist()
    label_names = ["entailment", "neutral", "contradiction"]

    return {name: float(pred) for pred, name in zip(prediction, label_names)}

def detect(document):
    sentences = nltk.tokenize.sent_tokenize(document, language='english')
    sentence_count = len(sentences)

    print(f'Split text into {sentence_count} sentences.')

    embeddings = embed_model.encode(sentences)
    pairs = get_closest_sentence_pairs(sentences, embeddings)

    p_contra_max = 0.0
    evidence = []

    for a, b, similarity in tqdm.tqdm(pairs):
        prediction = predict(a, b)
        p_contra = prediction['contradiction']
        p_contra_max = max(p_contra_max, p_contra)

        #print(f'Comparing:\n  A: {a}\n  B: {b}\n  Similarity: {similarity:.4f}')
        #print(f'    Prediction: {prediction}')
        #print()

        if p_contra > 0.75:
            evidence.append((a, b, p_contra))

    return p_contra_max, evidence

In [15]:
premise = "The sky is red"
hypothesis = "The sky is blue"

print(predict(premise, hypothesis))

{'entailment': 8.13603401184082e-05, 'neutral': 0.0005097389221191406, 'contradiction': 0.99951171875}


In [29]:
pos = False			# Test example from positive or negative set
example_id = None 	# '3489738232_6' # Specific example ID to test, or None for random

examples = dataset['pos' if pos else 'neg']

if example_id is not None:
	example = examples[example_id]
else:
	examples = list(examples.values())
	example = examples[random.randint(0, len(examples) - 1)]

print(f'Detecting contradictions in example {example["unique id"]}')
print()

print(example['text'])
print()

if pos:
	print(f'Evidence: {example["evidence"]}')
	print(f'Ref sentences: {example.get("ref sentences", [])}')

print()

p_contra, evidence = detect(example['text'])
p_contra, evidence

Detecting contradictions in example story_train_4081

 The novel has a complex plot, common in Collins’ work. In a Prologue, a selfish and ambitious man casts off his wife in order to marry a wealthier and better-connected woman, by taking advantage of a loophole in the marriage laws of Ireland.
The initial action takes place in the widowed Lady Lundie's house in Scotland. Geoffrey Delamayn has promised marriage to his lover Anne Silvester (governess to Lady Lundie's stepdaughter Blanche), who has incurred the enmity of her employer. The spendthrift Geoffrey is about to be disinherited, and wishes to escape from his promise and marry a wealthy wife. Nevertheless he is obliged to arrange a rendezvous with Anne, in the character of his wife, at an inn, and documents this in an exchange of notes with her. Subsequently, urgent matters force him to send his friend Arnold Brinkworth, Blanche's fiancé, to Anne in his place. To gain access to her, Arnold must ask for "his wife". Although nothi

100%|██████████| 32/32 [00:01<00:00, 22.48it/s]


(0.99853515625,
 [('Eventually Anne offers to reveal her relations with Geoffrey, even at the cost of her reputation – impressing Sir Patrick with her courageous and honourable behaviour.',
   'Geoffrey forces Hester to show him how to do the same to Anne.',
   0.98681640625),
  ('Geoffrey forces Hester to show him how to do the same to Anne.',
   'Eventually Anne offers to reveal her relations with Geoffrey, even at the cost of her reputation – impressing Sir Patrick with her courageous and honourable behaviour.',
   0.97216796875),
  ('By various stratagems he gets Anne to sleep in a suitably-placed bed.',
   'Geoffrey forces Hester to show him how to do the same to Anne.',
   0.98095703125),
  ('However he suffers a stroke when about to smother her, and while unconscious is throttled by Hester, who belatedly recognises the enormity of what she has been abetting.',
   'Hester inadvertently reveals to Geoffrey that she murdered her brutal and rapacious husband by dismantling part of t